# 5I — Incremental Regression Ensemble (Pretrained Prediction Averaging)

## Objective

Combine the four already-trained Stream I estimators:

1. SGDRegressor — squared error
2. SGDRegressor — Huber
3. SGDRegressor — epsilon-insensitive
4. MLPRegressor

This is a **regression averaging ensemble**. It is not soft voting (soft voting averages class probabilities) and it does not refit/retrain the underlying models. Each base notebook has already created test predictions in accident-count units. This notebook checks alignment, averages the four predictions equally, and saves the ensemble result.

## 0. Paths and input prediction files

In [ ]:
from pathlib import Path
import os, json, warnings, gc, time, copy, random, sys
import numpy as np
import pandas as pd

SEED = 34
random.seed(SEED)
np.random.seed(SEED)

ON_KAGGLE = Path('/kaggle/working').exists()
PROJECT_ROOT = Path('/kaggle/working/us_accidents_weekly_forecasting') if ON_KAGGLE else Path('./us_accidents_weekly_forecasting')
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
MODEL_DIR = PROJECT_ROOT / 'models'
REPORT_DIR = PROJECT_ROOT / 'reports'
for _p in (PROJECT_ROOT, ARTIFACT_DIR, MODEL_DIR, REPORT_DIR):
    _p.mkdir(parents=True, exist_ok=True)

# Set this only when more than one prior notebook output is attached as a Kaggle input.
ARTIFACT_INPUT_ROOT = None  # e.g. Path('/kaggle/input/us-accidents-weekly-stage-2')

def resolve_artifact(filename, required=True):
    candidates = []
    if ARTIFACT_INPUT_ROOT is not None:
        p = Path(ARTIFACT_INPUT_ROOT) / filename
        if p.exists():
            candidates.append(p)
    p = ARTIFACT_DIR / filename
    if p.exists():
        candidates.append(p)
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        candidates.extend(kaggle_input.rglob(filename))
    # Deduplicate resolved paths and prefer the most recently modified attached artifact.
    unique = {str(p.resolve()): p for p in candidates}
    candidates = list(unique.values())
    if not candidates:
        if required:
            raise FileNotFoundError(
                f'Could not locate {filename}. Add the preceding notebook output as a Kaggle input, '
                f'or set ARTIFACT_INPUT_ROOT. Local output searched: {ARTIFACT_DIR}'
            )
        return None
    chosen = max(candidates, key=lambda p: p.stat().st_mtime)
    if len(candidates) > 1:
        print(f'Found {len(candidates)} copies of {filename}; using newest: {chosen}')
    return chosen

print('Project root:', PROJECT_ROOT.resolve())
print('Seed:', SEED)

PREDICTION_FILES = {
    'SGD Squared Error': 'sgd_squared_test_predictions.csv',
    'SGD Huber': 'sgd_huber_test_predictions.csv',
    'SGD Epsilon-Insensitive': 'sgd_epsilon_insensitive_test_predictions.csv',
    'MLPRegressor': 'mlp_regressor_test_predictions.csv',
}

frames = {}
for label, filename in PREDICTION_FILES.items():
    path = resolve_artifact(filename)
    frame = pd.read_csv(path, parse_dates=['target_week_start'])
    needed = {'target_week_id','target_week_start','actual_count','predicted_count'}
    if not needed.issubset(frame.columns):
        raise ValueError(f'{filename} missing columns: {sorted(needed-set(frame.columns))}')
    if frame.target_week_id.duplicated().any():
        raise AssertionError(f'Duplicate target_week_id in {filename}')
    frames[label] = frame.sort_values('target_week_id').reset_index(drop=True)
    print(label, 'rows:', len(frame), '|', path)

## 1. Verify exact test-period alignment

In [ ]:
reference = next(iter(frames.values()))
reference_ids = reference.target_week_id.to_numpy()
reference_actuals = reference.actual_count.to_numpy(dtype=float)
for label, frame in frames.items():
    if not np.array_equal(frame.target_week_id.to_numpy(), reference_ids):
        raise AssertionError(f'Test week IDs differ for {label}. All base models must use identical test weeks.')
    if not np.allclose(frame.actual_count.to_numpy(dtype=float), reference_actuals, rtol=0, atol=1e-6):
        raise AssertionError(f'Actual target values differ for {label}.')
print('All four base prediction files are aligned on the same test weeks and actual counts.')

## 2. Equal-weight arithmetic mean (no retraining)

In [ ]:
prediction_matrix = np.column_stack([
    np.maximum(frames[label].predicted_count.to_numpy(dtype=float), 0.0)
    for label in PREDICTION_FILES
])
ensemble_prediction = np.mean(prediction_matrix, axis=1)
ensemble_prediction = np.maximum(ensemble_prediction, 0.0)

ensemble = reference[['target_week_id','target_week_start','actual_count']].copy()
ensemble['predicted_count'] = ensemble_prediction
ensemble['model'] = 'Incremental Mean Ensemble'
ensemble['ensemble_members'] = len(PREDICTION_FILES)
ensemble['ensemble_method'] = 'equal_weight_arithmetic_mean'

output_path = REPORT_DIR / 'incremental_ensemble_test_predictions.csv'
ensemble.to_csv(output_path, index=False)
manifest = {
    'ensemble_name':'Incremental Mean Ensemble',
    'members':list(PREDICTION_FILES.keys()),
    'weights':{name:1.0/len(PREDICTION_FILES) for name in PREDICTION_FILES},
    'method':'equal-weight arithmetic mean of pretrained base-model predictions',
    'base_models_retrained':False,
    'test_weeks':int(len(ensemble)),
    'prediction_file':str(output_path),
}
(REPORT_DIR/'incremental_ensemble_manifest.json').write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2))

## 3. Quick inspection

In [ ]:
display(ensemble.head(10))